# Interactive Optical Parameter Explorer

Sliders update the plot in real-time.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '10'

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from ipywidgets import FloatSlider, VBox, HBox, Output, interactive_output
import ipywidgets as widgets
from IPython.display import display

from fit_optics import FitConfig, load_and_bin_data, forward_model, load_ccd_geometry

In [2]:
# Load data and geometry once
config = FitConfig()
data = load_and_bin_data('data/visit_test_2025090600388_g_band.parquet', config)
geometry = load_ccd_geometry()
print(f"Loaded {len(data['detector'])} CCDs")

Loaded 180 CCDs


In [3]:
# Pre-compute observed quantities (don't change)
raw_dT = data['raw_T'] - np.nanmean(data['raw_T'])
raw_de1 = data['raw_e1'] - np.nanmean(data['raw_e1'])
raw_de2 = data['raw_e2'] - np.nanmean(data['raw_e2'])
obs_dT = data['T'] - np.nanmean(data['T'])
obs_de1 = data['e1'] - np.nanmean(data['e1'])
obs_de2 = data['e2'] - np.nanmean(data['e2'])
detectors = data['detector']

vmin_dT, vmax_dT = -1.5, 1.5
vmin_de, vmax_de = -0.10, 0.10

In [4]:
def plot_ccd_polygons(ax, dets, values, geom, cmap, vmin, vmax):
    """Plot CCDs as polygons with colors based on values."""
    patches = []
    colors = []
    for det_id, val in zip(dets, values):
        if np.isfinite(val) and det_id in geom:
            corners = geom[det_id]['corners']
            poly = Polygon(corners, closed=True)
            patches.append(poly)
            colors.append(val)
    if len(patches) == 0:
        return None
    collection = PatchCollection(patches, cmap=cmap, edgecolor='k', linewidth=0.3)
    collection.set_array(np.array(colors))
    collection.set_clim(vmin, vmax)
    ax.add_collection(collection)
    return collection

def make_plot(cam_dz, cam_dx, cam_dy, cam_rx, cam_ry,
              m2_dz, m2_dx, m2_dy, m2_rx, m2_ry,
              m1m3_b0, m1m3_b1, m1m3_b2, m1m3_b3, m1m3_b4,
              m2_b0, m2_b1, m2_b2):
    
    fit_names = [
        'cam_dz', 'cam_dx', 'cam_dy', 'cam_rx', 'cam_ry',
        'm2_dz', 'm2_dx', 'm2_dy', 'm2_rx', 'm2_ry',
        'm1m3_bend_0', 'm1m3_bend_1', 'm1m3_bend_2', 'm1m3_bend_3', 'm1m3_bend_4',
        'm2_bend_0', 'm2_bend_1', 'm2_bend_2'
    ]
    fit_values = np.array([
        cam_dz, cam_dx, cam_dy, cam_rx, cam_ry,
        m2_dz, m2_dx, m2_dy, m2_rx, m2_ry,
        m1m3_b0, m1m3_b1, m1m3_b2, m1m3_b3, m1m3_b4,
        m2_b0, m2_b1, m2_b2
    ])
    
    # Compute forward model
    chi2 = forward_model(fit_values, fit_names, data, config)
    pred = forward_model(fit_values, fit_names, data, config, return_full=True)
    
    res_dT = pred['dT'] - obs_dT
    res_de1 = pred['de1'] - obs_de1
    res_de2 = pred['de2'] - obs_de2
    
    # Plot
    fig, axes = plt.subplots(4, 3, figsize=(12, 14))
    lim = 350
    
    # Row 1: Raw observed
    axes[0, 0].scatter(data['raw_x'], data['raw_y'], c=raw_dT, s=1, cmap='seismic', vmin=vmin_dT, vmax=vmax_dT)
    axes[0, 0].set_title('Raw Observed dT'); axes[0, 0].set_aspect('equal'); axes[0, 0].set_xlim(-lim, lim); axes[0, 0].set_ylim(-lim, lim)
    axes[0, 1].scatter(data['raw_x'], data['raw_y'], c=raw_de1, s=1, cmap='seismic', vmin=vmin_de, vmax=vmax_de)
    axes[0, 1].set_title('Raw Observed de1'); axes[0, 1].set_aspect('equal'); axes[0, 1].set_xlim(-lim, lim); axes[0, 1].set_ylim(-lim, lim)
    axes[0, 2].scatter(data['raw_x'], data['raw_y'], c=raw_de2, s=1, cmap='seismic', vmin=vmin_de, vmax=vmax_de)
    axes[0, 2].set_title('Raw Observed de2'); axes[0, 2].set_aspect('equal'); axes[0, 2].set_xlim(-lim, lim); axes[0, 2].set_ylim(-lim, lim)
    
    # Row 2: Observed per CCD
    plot_ccd_polygons(axes[1, 0], detectors, obs_dT, geometry, 'seismic', vmin_dT, vmax_dT)
    axes[1, 0].set_title('Observed <dT> per CCD'); axes[1, 0].set_aspect('equal'); axes[1, 0].set_xlim(-lim, lim); axes[1, 0].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[1, 1], detectors, obs_de1, geometry, 'seismic', vmin_de, vmax_de)
    axes[1, 1].set_title('Observed <de1> per CCD'); axes[1, 1].set_aspect('equal'); axes[1, 1].set_xlim(-lim, lim); axes[1, 1].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[1, 2], detectors, obs_de2, geometry, 'seismic', vmin_de, vmax_de)
    axes[1, 2].set_title('Observed <de2> per CCD'); axes[1, 2].set_aspect('equal'); axes[1, 2].set_xlim(-lim, lim); axes[1, 2].set_ylim(-lim, lim)
    
    # Row 3: Predicted
    plot_ccd_polygons(axes[2, 0], detectors, pred['dT'], geometry, 'seismic', vmin_dT, vmax_dT)
    axes[2, 0].set_title('Predicted dT'); axes[2, 0].set_aspect('equal'); axes[2, 0].set_xlim(-lim, lim); axes[2, 0].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[2, 1], detectors, pred['de1'], geometry, 'seismic', vmin_de, vmax_de)
    axes[2, 1].set_title('Predicted de1'); axes[2, 1].set_aspect('equal'); axes[2, 1].set_xlim(-lim, lim); axes[2, 1].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[2, 2], detectors, pred['de2'], geometry, 'seismic', vmin_de, vmax_de)
    axes[2, 2].set_title('Predicted de2'); axes[2, 2].set_aspect('equal'); axes[2, 2].set_xlim(-lim, lim); axes[2, 2].set_ylim(-lim, lim)
    
    # Row 4: Residuals
    plot_ccd_polygons(axes[3, 0], detectors, res_dT, geometry, 'seismic', vmin_dT, vmax_dT)
    axes[3, 0].set_title('Residual dT'); axes[3, 0].set_aspect('equal'); axes[3, 0].set_xlim(-lim, lim); axes[3, 0].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[3, 1], detectors, res_de1, geometry, 'seismic', vmin_de, vmax_de)
    axes[3, 1].set_title('Residual de1'); axes[3, 1].set_aspect('equal'); axes[3, 1].set_xlim(-lim, lim); axes[3, 1].set_ylim(-lim, lim)
    plot_ccd_polygons(axes[3, 2], detectors, res_de2, geometry, 'seismic', vmin_de, vmax_de)
    axes[3, 2].set_title('Residual de2'); axes[3, 2].set_aspect('equal'); axes[3, 2].set_xlim(-lim, lim); axes[3, 2].set_ylim(-lim, lim)
    
    fig.suptitle(f'Chi2 = {chi2:.1f}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [5]:
# Create sliders
style = {'description_width': '100px'}
layout = widgets.Layout(width='300px')

# Camera DOF
cam_dz = FloatSlider(value=0, min=-200, max=200, step=5, description='cam_dz (µm)', style=style, layout=layout, continuous_update=False)
cam_dx = FloatSlider(value=0, min=-200, max=200, step=5, description='cam_dx (µm)', style=style, layout=layout, continuous_update=False)
cam_dy = FloatSlider(value=0, min=-200, max=200, step=5, description='cam_dy (µm)', style=style, layout=layout, continuous_update=False)
cam_rx = FloatSlider(value=0, min=-10, max=10, step=0.2, description='cam_rx (")', style=style, layout=layout, continuous_update=False)
cam_ry = FloatSlider(value=0, min=-10, max=10, step=0.2, description='cam_ry (")', style=style, layout=layout, continuous_update=False)

# M2 DOF
m2_dz = FloatSlider(value=0, min=-200, max=200, step=5, description='m2_dz (µm)', style=style, layout=layout, continuous_update=False)
m2_dx = FloatSlider(value=0, min=-200, max=200, step=5, description='m2_dx (µm)', style=style, layout=layout, continuous_update=False)
m2_dy = FloatSlider(value=0, min=-200, max=200, step=5, description='m2_dy (µm)', style=style, layout=layout, continuous_update=False)
m2_rx = FloatSlider(value=0, min=-10, max=10, step=0.2, description='m2_rx (")', style=style, layout=layout, continuous_update=False)
m2_ry = FloatSlider(value=0, min=-10, max=10, step=0.2, description='m2_ry (")', style=style, layout=layout, continuous_update=False)

# Bending modes
m1m3_b0 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m1m3_b0', style=style, layout=layout, continuous_update=False)
m1m3_b1 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m1m3_b1', style=style, layout=layout, continuous_update=False)
m1m3_b2 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m1m3_b2', style=style, layout=layout, continuous_update=False)
m1m3_b3 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m1m3_b3', style=style, layout=layout, continuous_update=False)
m1m3_b4 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m1m3_b4', style=style, layout=layout, continuous_update=False)
m2_b0 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m2_b0', style=style, layout=layout, continuous_update=False)
m2_b1 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m2_b1', style=style, layout=layout, continuous_update=False)
m2_b2 = FloatSlider(value=0, min=-2, max=2, step=0.1, description='m2_b2', style=style, layout=layout, continuous_update=False)

# Layout sliders
cam_box = VBox([widgets.HTML('<b>Camera</b>'), cam_dz, cam_dx, cam_dy, cam_rx, cam_ry])
m2_box = VBox([widgets.HTML('<b>M2</b>'), m2_dz, m2_dx, m2_dy, m2_rx, m2_ry])
bend_box = VBox([widgets.HTML('<b>Bending</b>'), m1m3_b0, m1m3_b1, m1m3_b2, m1m3_b3, m1m3_b4, m2_b0, m2_b1, m2_b2])

ui = HBox([cam_box, m2_box, bend_box])

# Interactive output - updates when slider released
out = interactive_output(make_plot, {
    'cam_dz': cam_dz, 'cam_dx': cam_dx, 'cam_dy': cam_dy, 'cam_rx': cam_rx, 'cam_ry': cam_ry,
    'm2_dz': m2_dz, 'm2_dx': m2_dx, 'm2_dy': m2_dy, 'm2_rx': m2_rx, 'm2_ry': m2_ry,
    'm1m3_b0': m1m3_b0, 'm1m3_b1': m1m3_b1, 'm1m3_b2': m1m3_b2, 'm1m3_b3': m1m3_b3, 'm1m3_b4': m1m3_b4,
    'm2_b0': m2_b0, 'm2_b1': m2_b1, 'm2_b2': m2_b2
})

display(ui, out)

Output()